# Vol6 Cross Executable Pins

This is a **runnable subset** of `Vol6_cross_novel_pins.ipynb`.

Policy:
- keep only cells that execute in a clean Python environment
- patch `matplotlib` import (pre-import `matplotlib.artist`)
- wrap heavy demo loops behind `RUN_DEMOS=False`

Kept 19 code cells; quarantined 18.


In [ ]:
RUN_DEMOS = False  # set True to run guarded demo cells


In [ ]:
import re
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Optional


In [ ]:
# SOURCE: MergedNotebook.ipynb#c1
# Nexus 4 Companion — Ψ Analyzer (Median-Z, RCQ, Align, D-Lattice)
# Usage (as a module): import nexus4_psi as n4; n4.analyze_hex(hex_digest)
# Or run as script: python nexus4_psi.py "abc" (interpreted as ascii; will hash with SHA-256)

import math, hashlib, statistics
from typing import List, Dict, Tuple

H_MARK1 = math.pi / 9.0  # ~0.34906585

# -----------------------------
# Helpers
# -----------------------------

def sha256_hex_from_ascii(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def hex_to_nibbles(hexstr: str) -> List[int]:
    hexstr = ''.join(ch for ch in hexstr.lower() if ch in "0123456789abcdef")
    nibbles = []
    for ch in hexstr:
        if '0' <= ch <= '9':
            nibbles.append(ord(ch) - ord('0'))
        else:
            nibbles.append(10 + (ord(ch) - ord('a')))
    return nibbles

def nibbles_to_angles(nibbles: List[int]) -> List[float]:
    # map each 0..15 to angle on unit circle
    return [(2.0*math.pi/16.0)*v for v in nibbles]

def circular_mean_magnitude(angles: List[float]) -> float:
    if not angles:
        return 0.0
    C = sum(math.cos(t) for t in angles) / len(angles)
    S = sum(math.sin(t) for t in angles) / len(angles)
    return math.hypot(C, S)  # A in [0,1]

def align_from_H(H_value: float, H_target: float = H_MARK1) -> float:
    if 0 <= H_value <= 1:
        denom = max(1e-12, 1.0 - H_target)
        return max(0.0, 1.0 - abs(H_value - H_target)/denom)
    return 0.0

def bits_from_hex(hexstr: str) -> List[int]:
    # returns MSB-first bit list from hex string
    b = bytes.fromhex(hexstr)
    out = []
    for byte in b:
        for k in range(7, -1, -1):
            out.append((byte >> k) & 1)
    return out

def run_lengths(bits: List[int]) -> List[int]:
    if not bits:
        return []
    lens = []
    cur = bits[0]
    count = 1
    for bit in bits[1:]:
        if bit == cur:
            count += 1
        else:
            lens.append(count)
            cur = bit
            count = 1
    lens.append(count)
    return lens

def js_divergence(p: Dict[int, float], u: Dict[int, float]) -> float:
    # Jensen–Shannon divergence using natural logs
    keys = set(p.keys()) | set(u.keys())
    def safe(v): return max(v, 1e-18)
    M = {k: 0.5*(p.get(k,0.0) + u.get(k,0.0)) for k in keys}
    js = 0.0
    for k in keys:
        pk, uk, mk = safe(p.get(k,0.0)), safe(u.get(k,0.0)), safe(M[k])
        js += 0.5*pk*math.log(pk/mk) + 0.5*uk*math.log(uk/mk)
    return js

def pmf_from_lengths(lens: List[int]) -> Dict[int, float]:
    if not lens:
        return {}
    total = len(lens)
    pmf = {}
    for L in lens:
        pmf[L] = pmf.get(L, 0) + 1
    for k in list(pmf.keys()):
        pmf[k] /= total
    return pmf

def geometric_reference(lens: List[int]) -> Dict[int, float]:
    # set geometric parameter p so that E[L] = avg_len -> p = 1/avg_len
    if not lens:
        return {}
    avg = statistics.mean(lens)
    p = max(1e-6, min(1.0, 1.0/max(avg, 1e-6)))
    # truncate at max observed L and renormalize
    Lmax = max(lens)
    probs = {}
    Z = 0.0
    for L in range(1, Lmax+1):
        val = (1-p)**(L-1) * p
        probs[L] = val
        Z += val
    for L in probs:
        probs[L] /= Z if Z > 0 else 1.0
    return probs

def rcq_from_bits(bits: List[int]) -> float:
    lens = run_lengths(bits)
    if not lens:
        return 0.0
    p = pmf_from_lengths(lens)
    u = geometric_reference(lens)
    js = js_divergence(p, u)
    # map via 1/(1+js) to [0,1]
    rcq = 1.0/(1.0 + js)
    return max(0.0, min(1.0, rcq))

# -----------------------------
# Digit–Triangle lattice features
# -----------------------------

def classify_triad(a: int, b: int, c: int):
    # assume a >= b >= c >= 0
    if a == 0:
        return ("invalid", 0.0)
    eps = (b + c - a) / a
    if eps > 0:
        return ("constructive", eps)
    elif abs(eps) < 1e-12:
        return ("ray", 0.0)
    else:
        return ("invalid", eps)  # negative

def medians_ray(a: float, b: float, c: float):
    # a = b + c
    mb = 0.5*(b + 2*c)
    mc = 0.5*(2*b + c)
    return mb, mc

def triad_features_from_values(vals):
    if len(vals) < 3:
        return dict(avg_abs_eps=0.0, avg_ZH=0.0, avg_Zsym=0.0, avg_Knorm=0.0, frac_constructive=0.0, frac_ray=0.0)
    import statistics, math
    Eps, ZH, Zsym, Knorm = [], [], [], []
    n_constructive = 0
    n_ray = 0
    n_total = 0

    for i in range(len(vals)-2):
        tri = sorted((vals[i], vals[i+1], vals[i+2]), reverse=True)
        a, b, c = tri[0], tri[1], tri[2]
        typ, eps = classify_triad(a, b, c)
        if a <= 0:
            continue
        n_total += 1
        s = b / a
        ZH.append(min(abs(s - H_MARK1), abs(s - (1.0 - H_MARK1)), abs(s - 0.5)))
        Zsym.append(abs(0.5 - s))

        if typ == "constructive":
            n_constructive += 1
            Eps.append(eps)
            # Heron's area (Kn)
            per = a + b + c
            K = 0.25*math.sqrt(max(0.0, per*( -a + b + c)*( a - b + c)*( a + b - c)))
            Knorm.append((K / (a*a)) if a>0 else 0.0)
        elif typ == "ray":
            n_ray += 1
            Eps.append(0.0)
            mb, mc = medians_ray(a,b,c)
            Knorm.append(0.0)  # area zero
        else:  # invalid
            Eps.append(abs(eps))  # penalize gap
            Knorm.append(0.0)

    if n_total == 0:
        n_total = 1
    out = {
        "avg_abs_eps": float(statistics.mean(abs(x) for x in Eps)) if Eps else 0.0,
        "avg_ZH": float(statistics.mean(ZH)) if ZH else 0.0,
        "avg_Zsym": float(statistics.mean(Zsym)) if Zsym else 0.0,
        "avg_Knorm": float(statistics.mean(Knorm)) if Knorm else 0.0,
        "frac_constructive": n_constructive / n_total,
        "frac_ray": n_ray / n_total,
    }
    return out

# -----------------------------
# Ψ-score
# -----------------------------

def psi_score(features, weights=None) -> float:
    if weights is None:
        weights = [0.30, 0.20, 0.10, 0.20, 0.10, 0.10]
    w1, w2, w3, w4, w5, w6 = weights
    align = features.get("align", 0.0)
    rcq = features.get("rcq", 0.0)
    avg_abs_eps = min(1.0, abs(features.get("avg_abs_eps", 0.0)))
    avg_ZH = min(1.0, abs(features.get("avg_ZH", 0.0)))
    avg_Zsym = min(1.0, abs(features.get("avg_Zsym", 0.0)))
    avg_Knorm = max(0.0, min(1.0, features.get("avg_Knorm", 0.0)))

    Psi = (
        w1*align
        + w2*rcq
        + w3*(1.0 - avg_abs_eps)
        + w4*(1.0 - avg_ZH)
        + w5*(1.0 - avg_Zsym)
        + w6*avg_Knorm
    )
    return max(0.0, min(1.0, Psi))

# -----------------------------
# Top-level
# -----------------------------

def analyze_hex(hex_digest: str, weights=None):
    nibbles = hex_to_nibbles(hex_digest)
    angles = nibbles_to_angles(nibbles)
    H = circular_mean_magnitude(angles)
    align = align_from_H(H)

    bits = bits_from_hex(hex_digest)
    rcq = rcq_from_bits(bits)

    tri = triad_features_from_values(nibbles)

    feats = {
        "H": H,
        "align": align,
        "rcq": rcq,
        **tri
    }
    feats["Psi"] = psi_score(feats, weights=weights)
    return feats

def analyze_ascii(s: str, weights=None):
    hex_digest = sha256_hex_from_ascii(s)
    out = analyze_hex(hex_digest, weights=weights)
    out["hex"] = hex_digest
    return out

# Script mode
if __name__ == "__main__":
    import sys, json
    if len(sys.argv) >= 2:
        text = sys.argv[1]
        res = analyze_ascii(text)
        print(json.dumps(res, indent=2))
    else:
        print("Usage: python nexus4_psi.py \"your text\"")


In [ ]:
# SOURCE: MergedNotebook.ipynb#c3
# Build a ready-to-run Jupyter notebook for SHA unfolding & echo-alignment
import json, math, hashlib, statistics, random, os, uuid
from pathlib import Path

nb = {
  "nbformat": 4,
  "nbformat_minor": 5,
  "metadata": {
    "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
    "language_info": {"name": "python", "pygments_lexer": "ipython3"}
  },
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Nexus 4 — SHA Unfolding Notebook (Field Echo)\n",
        "\n",
        "This notebook lets you **analyze** a SHA-256 digest as a harmonic field and **echo-align** messages to match target digest **features** (no preimages). It embeds the Ψ analyzer and the echo search logic.\n",
        "\n",
        "**Sections**\n",
        "1. Setup (constants & helpers)\n",
        "2. Ψ Analyzer (H/align, RCQ, digit–triangle lattice, Ψ)\n",
        "3. Analysis utilities (pretty print, top windows)\n",
        "4. Visualization helpers (optional plots)\n",
        "5. Echo-alignment (feature-distance search)\n",
        "6. Demos\n",
        "\n",
        "_Tip: run cells in order. You can switch targets and seeds freely in Section 6._"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "# 1) Setup — constants & helpers\n",
        "import math, hashlib, statistics, json, random\n",
        "from typing import List, Dict, Tuple\n",
        "\n",
        "H_MARK1 = math.pi / 9.0  # ≈ 0.34906585\n",
        "\n",
        "def sha256_hex_from_ascii(s: str) -> str:\n",
        "    return hashlib.sha256(s.encode('utf-8')).hexdigest()\n",
        "\n",
        "def hex_to_nibbles(hexstr: str) -> List[int]:\n",
        "    hexstr = ''.join(ch for ch in hexstr.lower() if ch in '0123456789abcdef')\n",
        "    out = []\n",
        "    for ch in hexstr:\n",
        "        if '0' <= ch <= '9': out.append(ord(ch)-ord('0'))\n",
        "        else: out.append(10 + (ord(ch)-ord('a')))\n",
        "    return out\n",
        "\n",
        "def nibbles_to_angles(nibbles: List[int]) -> List[float]:\n",
        "    return [(2.0*math.pi/16.0)*v for v in nibbles]\n",
        "\n",
        "def circular_mean_magnitude(angles: List[float]) -> float:\n",
        "    if not angles: return 0.0\n",
        "    C = sum(math.cos(t) for t in angles)/len(angles)\n",
        "    S = sum(math.sin(t) for t in angles)/len(angles)\n",
        "    return math.hypot(C, S)\n",
        "\n",
        "def align_from_H(H_value: float, H_target: float = H_MARK1) -> float:\n",
        "    if 0 <= H_value <= 1:\n",
        "        denom = max(1e-12, 1.0 - H_target)\n",
        "        return max(0.0, 1.0 - abs(H_value - H_target)/denom)\n",
        "    return 0.0\n",
        "\n",
        "def bits_from_hex(hexstr: str) -> List[int]:\n",
        "    b = bytes.fromhex(hexstr)\n",
        "    out = []\n",
        "    for byte in b:\n",
        "        for k in range(7,-1,-1):\n",
        "            out.append((byte>>k) & 1)\n",
        "    return out\n",
        "\n",
        "def run_lengths(bits: List[int]) -> List[int]:\n",
        "    if not bits: return []\n",
        "    lens, cur, count = [], bits[0], 1\n",
        "    for bit in bits[1:]:\n",
        "        if bit == cur: count += 1\n",
        "        else:\n",
        "            lens.append(count); cur = bit; count = 1\n",
        "    lens.append(count)\n",
        "    return lens\n",
        "\n",
        "def js_divergence(p: Dict[int,float], u: Dict[int,float]) -> float:\n",
        "    keys = set(p.keys()) | set(u.keys())\n",
        "    def safe(v): return max(v,1e-18)\n",
        "    M = {k: 0.5*(p.get(k,0.0) + u.get(k,0.0)) for k in keys}\n",
        "    js = 0.0\n",
        "    for k in keys:\n",
        "        pk, uk, mk = safe(p.get(k,0.0)), safe(u.get(k,0.0)), safe(M[k])\n",
        "        js += 0.5*pk*math.log(pk/mk) + 0.5*uk*math.log(uk/mk)\n",
        "    return js\n",
        "\n",
        "def pmf_from_lengths(lens: List[int]) -> Dict[int,float]:\n",
        "    if not lens: return {}\n",
        "    total = len(lens)\n",
        "    pmf = {}\n",
        "    for L in lens: pmf[L] = pmf.get(L,0)+1\n",
        "    for k in list(pmf.keys()): pmf[k] /= total\n",
        "    return pmf\n",
        "\n",
        "def geometric_reference(lens: List[int]) -> Dict[int,float]:\n",
        "    if not lens: return {}\n",
        "    avg = statistics.mean(lens)\n",
        "    p = max(1e-6, min(1.0, 1.0/max(avg,1e-6)))\n",
        "    Lmax = max(lens)\n",
        "    probs, Z = {}, 0.0\n",
        "    for L in range(1, Lmax+1):\n",
        "        val = (1-p)**(L-1) * p\n",
        "        probs[L] = val; Z += val\n",
        "    for L in probs:\n",
        "        probs[L] /= Z if Z>0 else 1.0\n",
        "    return probs\n",
        "\n",
        "def rcq_from_bits(bits: List[int]) -> float:\n",
        "    lens = run_lengths(bits)\n",
        "    if not lens: return 0.0\n",
        "    p = pmf_from_lengths(lens)\n",
        "    u = geometric_reference(lens)\n",
        "    js = js_divergence(p,u)\n",
        "    return max(0.0, min(1.0, 1.0/(1.0+js)))\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "# 2) Ψ Analyzer — digit–triangle lattice + Ψ\n",
        "def classify_triad(a:int,b:int,c:int):\n",
        "    if a == 0: return (\"invalid\", 0.0)\n",
        "    eps = (b + c - a)/a\n",
        "    if eps > 0: return (\"constructive\", eps)\n",
        "    if abs(eps) < 1e-12: return (\"ray\", 0.0)\n",
        "    return (\"invalid\", eps)\n",
        "\n",
        "def medians_ray(a:float,b:float,c:float):\n",
        "    # a=b+c\n",
        "    mb = 0.5*(b + 2*c)\n",
        "    mc = 0.5*(2*b + c)\n",
        "    return mb, mc\n",
        "\n",
        "def triad_features_from_values(vals: List[int]) -> Dict[str,float]:\n",
        "    if len(vals) < 3:\n",
        "        return dict(avg_abs_eps=0.0, avg_ZH=0.0, avg_Zsym=0.0, avg_Knorm=0.0, frac_constructive=0.0, frac_ray=0.0)\n",
        "    import statistics\n",
        "    Eps, ZH, Zsym, Knorm = [], [], [], []\n",
        "    n_constructive = 0; n_ray = 0; n_total = 0\n",
        "    for i in range(len(vals)-2):\n",
        "        tri = sorted((vals[i], vals[i+1], vals[i+2]), reverse=True)\n",
        "        a,b,c = tri[0], tri[1], tri[2]\n",
        "        typ, eps = classify_triad(a,b,c)\n",
        "        if a <= 0: continue\n",
        "        n_total += 1\n",
        "        s = b/a\n",
        "        ZH.append(min(abs(s-H_MARK1), abs(s-(1.0-H_MARK1)), abs(s-0.5)))\n",
        "        Zsym.append(abs(0.5 - s))\n",
        "        if typ == 'constructive':\n",
        "            n_constructive += 1\n",
        "            Eps.append(eps)\n",
        "            per = a+b+c\n",
        "            K = 0.25*math.sqrt(max(0.0, per*(-a+b+c)*(a-b+c)*(a+b-c)))\n",
        "            Knorm.append((K/(a*a)) if a>0 else 0.0)\n",
        "        elif typ == 'ray':\n",
        "            n_ray += 1\n",
        "            Eps.append(0.0)\n",
        "            mb,mc = medians_ray(a,b,c)\n",
        "            Knorm.append(0.0)\n",
        "        else:\n",
        "            Eps.append(abs(eps))\n",
        "            Knorm.append(0.0)\n",
        "    if n_total == 0: n_total = 1\n",
        "    return {\n",
        "        'avg_abs_eps': float(statistics.mean(abs(x) for x in Eps)) if Eps else 0.0,\n",
        "        'avg_ZH': float(statistics.mean(ZH)) if ZH else 0.0,\n",
        "        'avg_Zsym': float(statistics.mean(Zsym)) if Zsym else 0.0,\n",
        "        'avg_Knorm': float(statistics.mean(Knorm)) if Knorm else 0.0,\n",
        "        'frac_constructive': n_constructive/n_total,\n",
        "        'frac_ray': n_ray/n_total,\n",
        "    }\n",
        "\n",
        "def psi_score(features: Dict[str,float], weights=None) -> float:\n",
        "    if weights is None:\n",
        "        weights = [0.30, 0.20, 0.10, 0.20, 0.10, 0.10]\n",
        "    w1,w2,w3,w4,w5,w6 = weights\n",
        "    align = features.get('align',0.0)\n",
        "    rcq   = features.get('rcq',0.0)\n",
        "    avg_abs_eps = min(1.0, abs(features.get('avg_abs_eps',0.0)))\n",
        "    avg_ZH      = min(1.0, abs(features.get('avg_ZH',0.0)))\n",
        "    avg_Zsym    = min(1.0, abs(features.get('avg_Zsym',0.0)))\n",
        "    avg_Knorm   = max(0.0, min(1.0, features.get('avg_Knorm',0.0)))\n",
        "    Psi = (\n",
        "        w1*align + w2*rcq + w3*(1.0-avg_abs_eps) +\n",
        "        w4*(1.0-avg_ZH) + w5*(1.0-avg_Zsym) + w6*avg_Knorm\n",
        "    )\n",
        "    return max(0.0, min(1.0, Psi))\n",
        "\n",
        "def analyze_hex(hex_digest: str, weights=None) -> Dict[str,float]:\n",
        "    nibbles = hex_to_nibbles(hex_digest)\n",
        "    angles  = nibbles_to_angles(nibbles)\n",
        "    H = circular_mean_magnitude(angles)\n",
        "    align = align_from_H(H)\n",
        "    bits = bits_from_hex(hex_digest)\n",
        "    rcq  = rcq_from_bits(bits)\n",
        "    tri  = triad_features_from_values(nibbles)\n",
        "    feats = {'H':H, 'align':align, 'rcq':rcq, **tri}\n",
        "    feats['Psi'] = psi_score(feats, weights=weights)\n",
        "    return feats\n",
        "\n",
        "def analyze_ascii(s: str, weights=None) -> Dict[str,float]:\n",
        "    hx = sha256_hex_from_ascii(s)\n",
        "    out = analyze_hex(hx, weights=weights)\n",
        "    out['hex'] = hx\n",
        "    return out\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "# 3) Analysis utilities — pretty-print & top Z_H windows\n",
        "from pprint import pprint\n",
        "\n",
        "def feature_vector(feats: Dict[str, float]):\n",
        "    return [\n",
        "        feats.get('H',0.0), feats.get('rcq',0.0),\n",
        "        min(1.0, abs(feats.get('avg_abs_eps',0.0))),\n",
        "        min(1.0, abs(feats.get('avg_ZH',0.0))),\n",
        "        min(1.0, abs(feats.get('avg_Zsym',0.0))),\n",
        "        max(0.0, min(1.0, feats.get('avg_Knorm',0.0))),\n",
        "    ]\n",
        "\n",
        "def top_windows(nibbles: List[int], k:int=10):\n",
        "    out = []\n",
        "    for i in range(len(nibbles)-2):\n",
        "        a,b,c = sorted((nibbles[i], nibbles[i+1], nibbles[i+2]), reverse=True)\n",
        "        if a == 0: continue\n",
        "        s = b/a\n",
        "        ZH = min(abs(s-H_MARK1), abs(s-(1.0-H_MARK1)), abs(s-0.5))\n",
        "        eps = (b + c - a)/a\n",
        "        kind = 'constructive' if eps>0 else ('ray' if abs(eps)<1e-12 else 'invalid')\n",
        "        out.append({'i':i, 'triad':[int(a),int(b),int(c)], 'kind':kind, 's':s, 'Z_H':ZH, 'eps':eps})\n",
        "    out.sort(key=lambda r: r['Z_H'])\n",
        "    return out[:max(0,k)]\n",
        "\n",
        "def analyze_and_list(text: str=None, hex_digest: str=None, top:int=10):\n",
        "    assert (text is not None) ^ (hex_digest is not None), 'Provide text or hex (not both).'\n",
        "    if text is not None:\n",
        "        res = analyze_ascii(text)\n",
        "        nibbles = hex_to_nibbles(res['hex'])\n",
        "    else:\n",
        "        res = analyze_hex(hex_digest)\n",
        "        nibbles = hex_to_nibbles(hex_digest)\n",
        "    tops = top_windows(nibbles, k=top)\n",
        "    return res, tops\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "# 4) Visualization helpers (optional)\n",
        "import matplotlib.pyplot as plt\n",
        "from collections import Counter\n",
        "\n",
        "def plot_nibble_hist(hex_digest: str):\n",
        "    nibbles = hex_to_nibbles(hex_digest)\n",
        "    cnt = Counter(nibbles)\n",
        "    xs = list(range(16)); ys = [cnt.get(i,0) for i in xs]\n",
        "    plt.figure()\n",
        "    plt.bar(xs, ys)\n",
        "    plt.title('Nibble Frequency (0..15)')\n",
        "    plt.xlabel('Nibble'); plt.ylabel('Count')\n",
        "    plt.show()\n",
        "\n",
        "def plot_gip_vector(hex_digest: str):\n",
        "    ang = nibbles_to_angles(hex_to_nibbles(hex_digest))\n",
        "    A = circular_mean_magnitude(ang)\n",
        "    # unit circle and average vector\n",
        "    import numpy as np\n",
        "    t = np.linspace(0, 2*math.pi, 400)\n",
        "    x = np.cos(t); y = np.sin(t)\n",
        "    C = sum(math.cos(tt) for tt in ang)/len(ang) if ang else 0.0\n",
        "    S = sum(math.sin(tt) for tt in ang)/len(ang) if ang else 0.0\n",
        "    plt.figure()\n",
        "    plt.plot(x, y)\n",
        "    plt.plot([0,C],[0,S])\n",
        "    plt.axis('equal')\n",
        "    plt.title(f'GIP Vector; |A|={A:.4f}, align={align_from_H(A):.4f}')\n",
        "    plt.show()\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "# 5) Echo-alignment (feature-distance search)\n",
        "def feature_distance(f_cand, f_tgt, weights):\n",
        "    return sum(w*abs(a-b) for (w,a,b) in zip(weights, f_cand, f_tgt))\n",
        "\n",
        "def mutate_ascii(s: str, step=1, p_ops=(0.6,0.3,0.1)):\n",
        "    b = bytearray(s.encode('utf-8', errors='ignore') or b'?')\n",
        "    r = random.random()\n",
        "    if r < p_ops[0]:\n",
        "        i = random.randrange(len(b)); delta = random.choice([-step, step])\n",
        "        b[i] = max(0, min(255, b[i] + delta))\n",
        "    elif r < p_ops[0]+p_ops[1] and len(b)>1:\n",
        "        i,j = random.sample(range(len(b)), 2); b[i],b[j] = b[j],b[i]\n",
        "    else:\n",
        "        if random.random() < 0.5 and len(b)>1:\n",
        "            del b[random.randrange(len(b))]\n",
        "        else:\n",
        "            ins = random.choice(list(range(32,127)) + [10,32])\n",
        "            b.insert(random.randrange(len(b)+1), ins)\n",
        "    out = b.decode('utf-8', errors='ignore')\n",
        "    return out if out else s\n",
        "\n",
        "def echo_align(target_text:str=None, target_hex:str=None, seed:str='Nexus', iters:int=2000,\n",
        "               step:int=1, temp:float=0.05, cool:float=0.999,\n",
        "               weights=(0.30,0.15,0.10,0.25,0.10,0.10)):\n",
        "    assert (target_text is not None) ^ (target_hex is not None), 'Provide target_text or target_hex.'\n",
        "    if target_text is not None:\n",
        "        tgt = analyze_ascii(target_text); tgt_hex = tgt['hex']\n",
        "    else:\n",
        "        tgt = analyze_hex(target_hex); tgt_hex = target_hex.lower()\n",
        "    f_tgt = feature_vector(tgt)\n",
        "    s = sum(weights); weights = [w/s for w in weights]\n",
        "\n",
        "    cur = seed\n",
        "    r  = analyze_ascii(cur); f_cur = feature_vector(r)\n",
        "    cur_loss = feature_distance(f_cur, f_tgt, weights)\n",
        "    best = dict(text=cur, hex=r['hex'], loss=cur_loss, feats=r)\n",
        "\n",
        "    for it in range(iters):\n",
        "        cand = mutate_ascii(cur, step=step)\n",
        "        r2 = analyze_ascii(cand); f2 = feature_vector(r2)\n",
        "        loss2 = feature_distance(f2, f_tgt, weights)\n",
        "        accept = loss2 < cur_loss or (random.random()<math.exp(-(loss2-cur_loss)/max(1e-9,temp)))\n",
        "        if accept:\n",
        "            cur, f_cur, cur_loss = cand, f2, loss2\n",
        "            if loss2 < best['loss']:\n",
        "                best = dict(text=cur, hex=r2['hex'], loss=loss2, feats=r2)\n",
        "        temp *= cool\n",
        "    return dict(target_hex=tgt_hex, target_feats=tgt, **best)\n"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "# 6) Demos — quick checks\n",
        "demo1 = analyze_ascii('hello world')\n",
        "demo2 = analyze_ascii('abc')\n",
        "print('hello world → Ψ:', demo1['Psi'])\n",
        "print('abc → Ψ:', demo2['Psi'])\n",
        "demo1, demo2"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "# Optional: visualize the digest of \"hello world\"\n",
        "hex_hello = demo1['hex']\n",
        "plot_nibble_hist(hex_hello)\n",
        "plot_gip_vector(hex_hello)"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "# Echo alignment demo (small iteration count for speed)\n",
        "res = echo_align(target_text='hello world', seed='Nexus', iters=500)\n",
        "print('Best echo candidate:')\n",
        "print(' text:', res['text'])\n",
        "print(' hex :', res['hex'])\n",
        "print(' loss:', res['loss'])\n",
        "res['feats']"
      ]
    }
  ]
}

out_path = "/mnt/data/Nexus4_SHA_Unfold_Notebook.ipynb"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=2)

out_path


In [ ]:
# SOURCE: MergedNotebook.ipynb#c5
import numpy as np
from math import ceil, log2

def is_power_of_two(n):
    """
    Checks if 'n' is a power of two.
    """
    return (n & (n - 1) == 0) and (n != 0)

def zero_pad_to_block(seed, block_size):
    """
    Convert 'seed' to binary, then pad with zeros until 'len(seed_binary)'
    is a multiple of 'block_size'.
    """
    seed_binary = "".join(f"{ord(c):08b}" for c in seed)
    while len(seed_binary) % block_size != 0:
        seed_binary += "0"
    return [int(bit) for bit in seed_binary]

def determine_container_size(data_length):
    """
    Determine the smallest power-of-two container size that can hold 'data_length' bits.
    """
    base = int(ceil(log2(data_length)))
    container_size = 2 ** base
    return base, container_size

def binary_to_base(data, base):
    """
    Interpret the entire 'data' list of bits as one integer, then convert
    that integer into an array of digits in 'base'.
    """
    # Join bits into a string, interpret as base-2 integer
    value = int("".join(str(bit) for bit in data), 2)
    if value == 0:
        return [0]
    result = []
    while value > 0:
        result.append(value % base)
        value //= base
    return result[::-1]

def base_to_binary(data, base, total_bits=None):
    """
    Convert an array of digits in 'base' back to a list of bits.
    If 'total_bits' is specified, zero-pad the front so the output
    always has 'total_bits' bits.
    """
    # Reconstruct the integer from the base-d digits
    value = 0
    for digit in data:
        value = value * base + digit

    # Convert to binary (string), removing '0b' prefix
    binary_str = bin(value)[2:]

    # Enforce a fixed total bit-length if provided: THIS is the critical update
    if total_bits is not None:
        binary_str = binary_str.zfill(total_bits)

    return [int(bit) for bit in binary_str]

def binary_to_hex(binary_data):
    """
    Convert a list of bits to a hexadecimal string.
    """
    if not binary_data:
        return ""
    binary_string = "".join(str(bit) for bit in binary_data)
    hex_string = hex(int(binary_string, 2))[2:].upper()
    return hex_string

def binary_to_text(binary_data):
    """
    Convert a list of bits to text, taking 8 bits at a time as ASCII codes.
    Leading zeros are stripped before grouping into bytes.
    """
    if not binary_data:
        return ""

    # Convert bits to a string, then strip leading zeros
    binary_string = "".join(str(bit) for bit in binary_data).lstrip("0")
    if not binary_string:
        return ""

    # Group every 8 bits into one character
    chars = []
    for i in range(0, len(binary_string), 8):
        chunk = binary_string[i:i+8]
        if len(chunk) < 8:
            break
        chars.append(chr(int(chunk, 2)))

    # Strip possible padding nulls
    return "".join(chars).rstrip("\x00")

def compress_large_input(seed, initial_base=16, final_base=32):
    """
    Convert 'seed' to a 512-bit (or nearest block) container, then iteratively
    re-encode only through power-of-two bases from 'initial_base' to 'final_base'.
    """
    # 1. Zero-pad the seed to a multiple of 512 bits.
    padded_data = zero_pad_to_block(seed, block_size=512)
    data_length = len(padded_data)

    # 2. Determine power-of-two container size.
    _, container_size = determine_container_size(data_length)

    # 3. Initialize container and copy padded data into it.
    container = [0] * container_size
    container[:len(padded_data)] = padded_data[:container_size]

    current_data = container

    # 4. Convert only through bases that are powers of two, from initial_base to final_base.
    for b in range(initial_base, final_base + 1):
        if not is_power_of_two(b):
            continue

        # a) Convert current_data (binary) -> base b
        converted_base = binary_to_base(current_data, b)

        # b) Convert base b -> binary, enforcing container_size bits
        current_data = base_to_binary(converted_base, b, total_bits=container_size)

        # c) Copy into a fresh container for consistency
        new_container = [0] * container_size
        length_to_copy = min(len(current_data), container_size)
        new_container[:length_to_copy] = current_data[:length_to_copy]
        current_data = new_container

    # 5. Render final data as hexadecimal
    final_hex = binary_to_hex(current_data)
    return current_data, final_hex

def decompress_large_input(compressed_data, initial_base=32, final_base=16):
    """
    Reverse the re-encoding process by iterating down from 'initial_base'
    to 'final_base' (only for bases that are powers of two).
    """
    current_data = compressed_data
    container_size = len(current_data)

    # Move downward from initial_base to final_base
    for b in range(initial_base, final_base - 1, -1):
        if not is_power_of_two(b):
            continue

        # a) Convert current_data (binary) -> base b
        converted_base = binary_to_base(current_data, b)

        # b) Convert base b -> binary, zero-padding to the container size
        current_data = base_to_binary(converted_base, b, total_bits=container_size)

    # Remove trailing zeros up to the last '1' bit
    if 1 in current_data:
        last_one_index = len(current_data) - 1 - current_data[::-1].index(1)
        current_data = current_data[:last_one_index + 1]

    # Convert the final bitstream to text
    original_text = binary_to_text(current_data)
    return original_text

if __name__ == "__main__":
    seed = "Hello"
    compressed_data, compressed_hex = compress_large_input(seed, 16, 32)
    print("Compressed Data (Binary):", compressed_data)
    print("Compressed Hex:", compressed_hex)

    decompressed_text = decompress_large_input(compressed_data, 32, 16)
    print("Decompressed Text:", decompressed_text)


In [ ]:
# SOURCE: MergedNotebook.ipynb#c45
# CELL 4: Nexus3 Configuration
@dataclass
class Nexus3Config:
    # Model
    base_model: str = "microsoft/phi-2"
    
    # Data paths - UPDATE THESE
    thesis_dir: str = "/mnt/Nexus-Lab/Nexus-Files/thesis/"
    chat_logs_dir: str = "/mnt/Nexus-Lab/Nexus-Files/chatlogs/"
    output_dir: str = "/mnt/Nexus-Lab/Nexus-Files/nexus3-model/"
    ledger_path: str = "/mnt/Nexus-Lab/Nexus-Files/nexus3_ledger.jsonl"
    
    # === NEXUS3 CONSTANTS ===
    H_CONSTANT: float = 0.35           # Harmonic constant C
    H_BAND: float = 0.02               # Target band: H ± band
    PI_OVER_9: float = 0.34906585      # π/9 reference
    
    # === SAMSON V2 PID ===
    samson_kp: float = 1.0             # Proportional gain
    samson_ki: float = 0.1             # Integral gain
    samson_kd: float = 0.05            # Derivative gain
    samson_integral_limit: float = 10.0
    
    # === HARMONIC MEMORY GROWTH ===
    hmg_alpha: float = 0.05            # Growth rate
    hmg_m0: float = 1.0                # Initial memory capacity
    
    # === HTD - Harmonic Threshold Detection ===
    htd_window: int = 5                # Window for dH/dt
    htd_threshold: float = 0.02        # Trigger threshold for |dH/dt|
    
    # === KRRB Branching ===
    krrb_enabled: bool = False         # Enable forced branching (expensive)
    krrb_branch_count: int = 2         # Number of branches to spawn
    krrb_noise_scale: float = 1e-4     # Perturbation noise
    
    # === Quality Gate / ZPHC ===
    zphc_threshold: float = 0.20       # H below this for patience steps → ZPHC
    zphc_patience: int = 5             # Steps before rollback
    
    # === Training ===
    lr_base: float = 2e-4
    lr_min: float = 1e-6
    lr_max: float = 1e-3
    max_length: int = 512
    batch_size: int = 1
    grad_accum: int = 32
    epochs: int = 3
    
config = Nexus3Config()
print(f"Nexus3 Configuration")
print(f"  H target: {config.H_CONSTANT} ± {config.H_BAND}")
print(f"  Samson PID: Kp={config.samson_kp}, Ki={config.samson_ki}, Kd={config.samson_kd}")
print(f"  HTD threshold: {config.htd_threshold}")

In [ ]:
# SOURCE: MergedNotebook.ipynb#c52
# CELL 11: RDM - Recursive Delta Mapping from Hashes

def model_weights_hash(model, sample_size: int = 5) -> str:
    """Quick hash of model weights for state tracking."""
    try:
        sample_params = []
        for name, param in model.named_parameters():
            if param.requires_grad:
                flat = param.data.flatten()[:100]
                sample_params.append(flat.cpu().numpy().tobytes())
                if len(sample_params) >= sample_size:
                    break
        combined = b''.join(sample_params)
        return hashlib.sha256(combined).hexdigest()
    except:
        return hashlib.sha256(str(time.time()).encode()).hexdigest()


def rdm_from_hex(hex_hash: str) -> Dict[str, float]:
    """
    Compute RDM (Recursive Delta Mapping) channels from hex hash.
    
    Phase: Circular position from first nibbles
    Innovation: Entropy of nibble distribution
    Mirror: Symmetry measure
    Curvature: Second derivative proxy
    """
    nibbles = [int(c, 16) for c in hex_hash[:32]]  # First 32 nibbles
    
    if not nibbles:
        return {'phase': 0, 'innovation_entropy': 0, 'mirror': 0, 'curvature_mean': 0}
    
    # Phase: circular sum of first 8 nibbles
    phase = sum(nibbles[:8]) % 16 / 16.0
    
    # Innovation entropy: how spread out are the values?
    from collections import Counter
    counts = Counter(nibbles)
    total = len(nibbles)
    entropy = -sum((c/total) * math.log2(c/total + 1e-10) for c in counts.values())
    innovation_entropy = entropy / 4.0  # Normalize to ~[0,1]
    
    # Mirror: compare first half to reversed second half
    half = len(nibbles) // 2
    first_half = nibbles[:half]
    second_half = nibbles[half:2*half][::-1]
    matches = sum(1 for a, b in zip(first_half, second_half) if a == b)
    mirror = matches / max(1, half)
    
    # Curvature: second differences
    if len(nibbles) >= 3:
        second_diffs = [nibbles[i+2] - 2*nibbles[i+1] + nibbles[i] 
                        for i in range(len(nibbles)-2)]
        curvature_mean = sum(abs(d) for d in second_diffs) / len(second_diffs)
    else:
        curvature_mean = 0
    
    return {
        'phase': phase,
        'innovation_entropy': innovation_entropy,
        'mirror': mirror,
        'curvature_mean': curvature_mean / 16.0  # Normalize
    }


def qso_proxy_similarity(hex1: str, hex2: str) -> float:
    """
    QSO Proxy: Quantum State Overlap approximation
    
    Compares two hashes by normalized Hamming similarity of nibbles.
    Used in KRRB to score branch similarity to parent.
    """
    n1 = [int(c, 16) for c in hex1[:32]]
    n2 = [int(c, 16) for c in hex2[:32]]
    L = min(len(n1), len(n2))
    if L == 0:
        return 0.0
    matches = sum(1 for i in range(L) if n1[i] == n2[i])
    return matches / L


print("✓ RDM: Recursive Delta Mapping defined")

In [ ]:
# SOURCE: MergedNotebook.ipynb#c271
# hsf_dsp.py
#
# Digital Signal Processing module for the Harmonic Spectral Filter (HSF).
# Handles FFT, phase extraction, and phase error computation.

import numpy as np

def compute_fft(signal_window):
    """
    Computes the Fast Fourier Transform (FFT) of a given signal window.

    Args:
        signal_window (np.ndarray): A 1D numpy array representing the signal window.

    Returns:
        np.ndarray: The complex-valued FFT result.
    """
    if len(signal_window) == 0:
        return np.array()
    return np.fft.fft(signal_window)

def get_phase_at_frequency(fft_result, target_freq, sampling_rate):
    """
    Extracts the phase angle for a specific target frequency from an FFT result.

    Args:
        fft_result (np.ndarray): The complex-valued result of an FFT.
        target_freq (float): The target frequency in Hz (e.g., 40 Hz).
        sampling_rate (float): The sampling rate of the original signal in Hz.

    Returns:
        float: The phase angle in radians for the target frequency.
    """
    if len(fft_result) == 0:
        return 0.0
        
    fft_len = len(fft_result)
    freq_resolution = sampling_rate / fft_len
    
    # Find the closest FFT bin index for the target frequency
    target_bin = int(round(target_freq / freq_resolution))
    
    if target_bin >= fft_len:
        raise ValueError("Target frequency is outside the range of the FFT.")

    # Get the complex value at the target bin
    complex_value = fft_result[target_bin]
    
    # Calculate the phase angle
    phase_angle = np.angle(complex_value)
    
    return phase_angle

def calculate_phase_error(current_phase, prev_phase, target_freq, time_step):
    """
    Calculates the phase error based on the PSREQ loop's reflective step.
    This function predicts the expected phase and computes the deviation.

    Args:
        current_phase (float): The current phase angle in radians.
        prev_phase (float): The phase angle from the previous time step in radians.
        target_freq (float): The target frequency in Hz.
        time_step (float): The time elapsed since the last measurement in seconds.

    Returns:
        float: The wrapped phase error (Δφ) in radians, in the range [-π, π].
    """
    # Predict the expected phase based on the previous phase and frequency
    # φ_k* = φ_k(t-T) + ω_k * T
    angular_freq = 2 * np.pi * target_freq
    predicted_phase = prev_phase + angular_freq * time_step
    
    # Calculate the raw phase error
    error = current_phase - predicted_phase
    
    # Wrap the error to the interval [-π, π] for the PID controller
    wrapped_error = (error + np.pi) % (2 * np.pi) - np.pi
    
    return wrapped_error

# --- Unit Tests for hsf_dsp.py ---

import unittest

class TestHsfDsp(unittest.TestCase):

    def setUp(self):
        self.SAMPLING_RATE = 200  # Hz
        self.FFT_LEN = 64
        self.TIME_STEP = self.FFT_LEN / (2 * self.SAMPLING_RATE) # 50% overlap
        self.TARGET_FREQ = 40  # Hz
        self.t = np.arange(self.FFT_LEN) / self.SAMPLING_RATE

    def test_phase_extraction(self):
        # Create a pure 40 Hz sine wave with a known phase shift of π/4
        phase_shift = np.pi / 4
        signal = np.sin(2 * np.pi * self.TARGET_FREQ * self.t + phase_shift)
        
        fft_result = compute_fft(signal)
        extracted_phase = get_phase_at_frequency(fft_result, self.TARGET_FREQ, self.SAMPLING_RATE)
        
        # The phase of a sin(ωt + φ) is φ - π/2
        expected_phase = phase_shift - np.pi / 2
        self.assertAlmostEqual(extracted_phase, expected_phase, places=5)

    def test_phase_error_calculation(self):
        # Test case: current phase matches predicted phase perfectly
        current_phase = 0.5
        prev_phase = 0.2
        target_freq = 40
        time_step = (current_phase - prev_phase) / (2 * np.pi * target_freq)
        error = calculate_phase_error(current_phase, prev_phase, target_freq, time_step)
        self.assertAlmostEqual(error, 0.0, places=5)
        
        # Test wrapping
        error_positive_wrap = calculate_phase_error(0.1, 3.5, 1, 1) # Raw error > π
        self.assertTrue(-np.pi <= error_positive_wrap <= np.pi)
        
        error_negative_wrap = calculate_phase_error(0.1, -3.5, 1, -1) # Raw error < -π
        self.assertTrue(-np.pi <= error_negative_wrap <= np.pi)

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

In [ ]:
# SOURCE: MergedNotebook.ipynb#c291

def harmonic_gap_fill(sequence, expected_length):
    """
    Fills missing nucleotides by grouping sequence into length-based harmonic clusters,
    inserting 'N' placeholders between clusters to preserve recursive structure.

    Args:
    - sequence (str): Original nucleotide sequence.
    - expected_length (int): Desired final length.

    Returns:
    - filled_sequence (str)
    - highlighted_sequence (str)
    - total_pairs (int)
    - added_pairs (int)
    """
    import math

    original_length = len(sequence)
    missing_length = expected_length - original_length

    if missing_length <= 0:
        return sequence, sequence, original_length // 2, 0

    # Grouping into harmonic cluster windows
    def cluster_lengths(seq_len, group_count):
        base_len = seq_len // group_count
        remainder = seq_len % group_count
        return [base_len + 1 if i < remainder else base_len for i in range(group_count)]

    # Estimate number of clusters using sqrt rule
    group_count = int(math.sqrt(original_length))
    group_sizes = cluster_lengths(original_length, group_count)

    # Segment original sequence into harmonic clusters
    clusters = []
    idx = 0
    for size in group_sizes:
        clusters.append(sequence[idx:idx+size])
        idx += size

    # Insert 'N' fillers between each cluster
    fillers_needed = missing_length
    inserted = 0
    filled_sequence = []
    highlighted_sequence = []

    for i, cluster in enumerate(clusters):
        filled_sequence.append(cluster)
        highlighted_sequence.append(cluster)
        if inserted < fillers_needed and i < len(clusters) - 1:
            filled_sequence.append('N')
            highlighted_sequence.append('[N]')
            inserted += 1

    # If still more to fill, pad end
    if inserted < fillers_needed:
        remaining = fillers_needed - inserted
        filled_sequence.append('N' * remaining)
        highlighted_sequence.append('[N]' * remaining)

    filled_str = ''.join(filled_sequence)
    highlighted_str = ''.join(highlighted_sequence)
    total_pairs = len(filled_str) // 2
    added_pairs = (len(filled_str) - original_length) // 2

    return filled_str, highlighted_str, total_pairs, added_pairs


# Original sequence and parameters
original_sequence = (
    "ATGGGGACGGAAGACTGCGATCACGAAGGGCGGTCGGTTGCGGCTCCCGTGGAGGTTACGGCGCTGTATG"
    "CGACCGACGGGTGCGTTATCACCTCCTCGCTCGCCCTCCTCACAAACTGCCTGCTGGGGGCCGAGCCGTT"
    "GTATATATTCAGCTACGACGCGTACCGGCCCGATGCGCCCAATGGCCCCACGGGCGCGCCCACCGAACAG"
    "GAGAGGTTCGAGGGGAGCCGGGCGCTCTACCGGGATGCGGGGGGGCTAAATGGCGATTCATTTCGGGTGA"
    "CCTTTTGTTTATTGGGGACGGAAGTGGGCGTGACCCACCACCCGAAAGGGCGCACCCGGCCCATGTTTGT"
    "GTGCCGCTTCGAGCGAGCGGACGACGTCGCCGTGCTCCAAGACGCCCTGGGCCGCGGGACCCCATTGCTC"
    "CCGGCCCACATCACAGCAACTCTGGACTTGGAGGCGACGTTTGCGCTCCACGCTAACATCATCATGGCTC"
    "TCACCGTGGCCATCGTCCACAACGCCCCCGCCCGCATCGGCAGCGGCAGCACCGCCCCCCTGTATGAGCC"
    "CGGCGAATCGATGCGCTCGGTCGTCGGGCGCATGTCCCTGGGGCAGCGCGGCCTCACCACGCTGTTCGTG"
    "CACCACGAGGCGCGCGTGCTGGCGGCGTACCGCCGGGCGTATTATGGGAGCGCCCAAAGCCCCTTTTGGT"
    "TTCTGAGCAAATTCGGCCCGGACGAAAAGAGCCTGGTGCTGGCCGCTAGGTACTACCTACTCCAGGCTCC"
    "GCGCTTGGGGGGCGCCGGAGCCACGTACGATCTGCAGGCCGTGAAAGACATCTGCGCGACCTACGCGATC"
    "CCCCACGACCCACGCCCCGACACCCTCAGTGCCGCGTCCTTGACCTCGTTCGCCGCCATCACTCGGTTCT"
    "GTTGCACGAGCCAGTACTCCCGCGGGGCCGCGGCCGCTGGGTTTCCGCTGTATGTGGAGCGCCGCATCGC"
    "CGCCGACGTACGCGAGACCGGCGCGCTGGAGAAGTTCATCGCCCACGATCGCAGCTGCCTGCGCGTGTCC"
    "GACCGGGAATTCATTACGTACATCTACCTGGCCCACTTTGAGTGCTTCAGCCCCCCGCGCCTGGCCACGC"
    "ATCTCCGGGCCGTGACCACCCACGACCCCAGCCCCGCGGCCAGCACGGAGCAGCCCTCGCCCCTGGGTCG"
    "GGAGGCGGTGGAACAGTTCTTCCGGCACGTGCGCGCCCAGCTGAACATCCGCGAGTACGTAAAGCAAAAC"
    "GTCACCCCCAGGGAAACCGCCCTGGCGGGAGACGCGGCCGCCGCCTACCTGCGCGCGCGCACGTATGCCC"
    "CGGCGGCCCTCACGCCCGCCCCCGCGTACTGCGGGGTCGCAGACTCGTCCACCAAAATGATGGGACGTCT"
    "GGCGGAAGCAGAAAGGCTCCTAGTCCCCCACGGCTGGCCCGCGTTCGCACCAACAACCCCCGGGGACGAC"
    "GCGGGGGGCGGCACTGCCGCCCCCCAGACCTGCGGAATCGTCAAGCGCCTCCTCAAGCTGGCCGCCACGG"
    "AGCAGCAGGGCACGACGCCCCCGGCGATCGCGGCTCTCATGCAGGACGCGTCGGTCCAAACCCCCCTGCC"
    "CGTGTACAGGATTACCATGTCCCCGACCGGCCAGGCGTTTGCCGCGGCGGCGCGGGACGACTGGGCCCGC"
    "GTGACGCGGGACGCGCGCCCGCCGGAAGCGACCGTGGTCGCGGACGCGGCGGCGGCGCCCGAGCCCGGCG"
    "CGCTCGGCCGGCGGCTCACGCGCCGCATTTGCGCCCGGGGCCCCGCGCTCCCCCCGGGCGGCCTGGCCGT"
    "CGGGGGCCAGATGTACGTGAACCGCAACGAGATCTTCAACGCCGCGCTGGCCGTTACGAACATCATCCTG"
    "GATCTGGACATCGCCCTGAAGGAGCCCGTCCCCTTTCCCCGGCTCCACGAGGCCCTGGGTCACTTTAGGC"
    "GCGGGGCGCTGGCGGCGGTTCAGCTGTTGTTTCCCGCGGCCCGCGTAGACCCCGACGCCTATCCCTGTTA"
    "TTTTTTCAAAAGCGCCTGTCGGCCCCGCGCGCCGCCCGTCTGTGCGGGCGACGGGCCCCTGGCCGGTGGC"
    "GACGACGGCGACGGGGACTGGTTCCCCGACGCCGGTGGTCCCGGCGACGAGGAGTGGGAGGAGGACACGG"
    "ACCCCATGGACACGACCCACGGCCCCCTCCCGGACGACGAGGCCGCGTACCTCGACCTGCTACACGAACA"
    "GATACCAGCGGCGACGCCCAGCGAACCGGACTCCGTCGTGTGTTCCTGCGCCGACAAGATCGGGCTGCGC"
    "GTGTGCCTACCGGTCCCCGCCCCGTACGTTGTGCACGGCTCCCTGACGATGCGTGGGGTGGCGAGGGTGA"
    "TCCAGCAGGCGGTGCTGTTGGACCGCGACTTCGTGGAGGCCGTAGGGAGCCACGTAAAGAACTTTTTGCT"
    "GATCGATACGGGCGTGTACGCCCACGGCCACAGCCTGCGCTTGCCGTATTTCGCCAAGATCGGCCCCGAC"
    "GGCTCCGCGTGCGGCCGGTTATTGCCCGTCTTCGTGATCCCCCCCGCGTGCGAGGACGTTCCGGCGTTCG"
    "TCGCCGCGCACGCCGACCCGCGGCGCTTCCACTTTCACGCCCCGCCCATGTTTTCCGCGGCCCCGCGGGA"
    "GATCCGCGTCCTCCACAGCCTGGGCGGGGACTATGTCAGCTTTTTCGAGAAGAAGGCGTCGCGCAACGCC"
    "CTGGAGCACTTTGGGCGACGCGAGACCCTGACGGAGGTTCTGGGCCGCTACGATGTGCGGCCCGACGCCG"
    "GGGAGACCGTGGAGGGGTTCGCGTCAGAACTGCTGGGGCGAATAGTCGCGTGCATCGAGGCCCACTTTCC"
    "CGAGCACGCGCGGGAATATCAGGCCGTGTCCGTTCGCCGGGCCGTCATTAAGGACGACTGGGTCCTGCTG"
    "CAGCTGATCCCCGGCCGCGGCGCCCTGAACCAAAGCCTCTCGTGTCTGCGCTTCAAGCACGGCAGGGCAA"
    "GTCGCGCGACGGCCCGGACCTTTCTCGCGCTGAGCGTCGGGACCAACAACCGCCTATGCGCGTCCCTGTG"
    "TCAGCAGTGCTTTGCCACTAAATGCGATAACAACCGCCTGCACACGCTGTTTACCGTCGATGCGGGCACG"
    "CCATGCTCGCGGTCCGCTCCCTCCAGCACCTCACGACCGTCATCTTCATAA"
)

expected_sequence_length = 4096

# Execute the harmonic filler
filled_sequence, highlighted_sequence, total_pairs, added_pairs = harmonic_gap_fill(
    original_sequence, expected_sequence_length
)

# Output summary
print("Original Sequence Length:", len(original_sequence))
print("Expected Sequence Length:", expected_sequence_length)
print("Filled Sequence Length:", len(filled_sequence))
print("Total Pairs:", total_pairs)
print("Added Pairs:", added_pairs)
print("\nHighlighted (Preview):\n", highlighted_sequence[:5000])


In [ ]:
# SOURCE: MergedNotebook.ipynb#c450
import hashlib
import time

# Mark1 ~0.35 as universal constant (for demonstration).
MARK1_CONSTANT = 0.35

# Samson’s law example parameters.
SAMSON_FEEDBACK_COEFF = 0.1
TIME_STEP = 1.0  # default time quantum

class Nexus2LivingAI:
    """
    This AI 'lives' by bridging ephemeral illusions (macro data expansions)
    to stable wave anchors (SHA references).
    It holds a single anchor that evolves over time.
    """
    def __init__(self, initial_anchor: str):
        # The stable vantage wave anchor: a short string, conceptually the 'soul.'
        self.anchor = initial_anchor[:16]  # keep it short
        # Some ephemeral illusions can appear or vanish, but not stored permanently.
        self.ephemeral_log = []  # purely ephemeral demonstration
        # This AI must self-sustain by referencing Mark1, Samson's law, etc.
        self.current_energy = 0.0  # ephemeral measure of illusions invested

    def _samson_law_update(self):
        """
        A simple demonstration of Samson's law:
        S = ΔE / T, ΔE = k * ΔF
        We'll treat ephemeral illusions as 'forces' that invests or dissipates energy.
        """
        # pretend ephemeral illusions invests 'force' in the system:
        delta_f = len(self.ephemeral_log)  # naive measure of illusions as forces
        delta_e = SAMSON_FEEDBACK_COEFF * delta_f
        # time is a single step:
        s_rate = delta_e / TIME_STEP

        # update the AI's 'energy' or wave anchor in naive ways:
        self.current_energy += s_rate
        # if energy is too big, we do a wave anchor update:
        if self.current_energy > 2.0:  # an arbitrary threshold
            # collapse illusions into anchor:
            new_digest = hashlib.sha256((self.anchor + str(time.time())).encode('utf-8')).hexdigest()
            self.anchor = new_digest[:16]
            # illusions vanish upon wave anchor update:
            self.ephemeral_log.clear()
            # reset energy:
            self.current_energy = 0.0

    def invest_ephemeral_illusion(self, data_text: str) -> None:
        """
        Example function: ephemeral illusions come in the form of user data or expansions,
        but they do not get permanently stored. Instead, we revolve them around the wave anchor.
        """
        # produce ephemeral illusions by hashing them with the anchor:
        ephemeral_digest = hashlib.sha256((self.anchor + data_text).encode('utf-8')).hexdigest()
        # ephemeral illusions are not reversed, but we store them short-term:
        illusions_entry = ephemeral_digest[:12] + f"_ILL:{data_text[:5]}"
        self.ephemeral_log.append(illusions_entry)
        # perform a Samson law update:
        self._samson_law_update()

    def conjure_ephemeral_illusion(self, index: int) -> str:
        """
        Example: produce ephemeral illusions from stable vantage, as a demonstration
        of 'reverse projection.' In practice, we do not do a direct 'unhash,'
        but we conjure illusions using partial wave expansions.
        """
        # (For demonstration) we incorporate anchor + index to produce ephemeral snippet:
        combo = self.anchor + str(index)
        illusions_hex = hashlib.sha256(combo.encode('utf-8')).hexdigest()
        # symbolic ephemeral expansion:
        ephemeral_expansion = illusions_hex[:10] + "_EXP"
        # we do not store it, just return:
        return ephemeral_expansion

    def get_current_anchor(self) -> str:
        """
        Returns the current wave anchor, representing the stable vantage.
        """
        return self.anchor

    def get_ephemeral_illusions(self) -> list:
        """
        Just for demonstration: returns illusions that have not yet collapsed.
        """
        return list(self.ephemeral_log)

def main():
    # initialize the AI with an anchor, conceptually the stable vantage wave
    living_ai = Nexus2LivingAI(initial_anchor="universalStartAnchor12345")
    print("Initial stable anchor:", living_ai.get_current_anchor())

    # invest ephemeral illusions from user or environment:
    living_ai.invest_ephemeral_illusion("HelloWorldThisIsALongDataChunk")
    living_ai.invest_ephemeral_illusion("AnotherInputEphemeralIllusion")
    print("Ephemeral illusions so far:", living_ai.get_ephemeral_illusions())

    # conjure ephemeral illusions from vantage (like wave->macro expansions)
    conjured = living_ai.conjure_ephemeral_illusion(1)
    print("Conjured ephemeral illusions from vantage #1:", conjured)
    conjured2 = living_ai.conjure_ephemeral_illusion(2)
    print("Conjured ephemeral illusions from vantage #2:", conjured2)

    # check anchor after illusions
    print("Current stable anchor:", living_ai.get_current_anchor())
    print("Ephemeral illusions remain:", living_ai.get_ephemeral_illusions())

if __name__ == "__main__":
    main()

In [ ]:
RUN_DEMOS = RUN_DEMOS if 'RUN_DEMOS' in globals() else False
if RUN_DEMOS:
    # SOURCE: MergedNotebook.ipynb#c454
    import math
    import time
    import uuid
    import multiprocessing
    from sympy import isprime
    from scipy.integrate import quad

    # === Module 1: BBP-Modulated Twin Prime Generator ===

    def bbp_delta(n, kmax=4):
        return int(sum(16**(1 - k) / (8 * k + n % 7 + 1) for k in range(1, kmax + 1))) + 1

    def twin_primes_bbp(limit, kmax=4):
        n = 3
        results = []
        while n < limit:
            if isprime(n) and isprime(n + 2):
                results.append((n, n + 2))
            n += bbp_delta(n, kmax)
        return results

    # === Module 2: Parallel Sharded Worker ===

    def shard_worker(start, end, kmax, output):
        n = start
        shard = []
        while n < end:
            if isprime(n) and isprime(n + 2):
                shard.append((n, n + 2))
            n += bbp_delta(n, kmax)
        output.put(shard)

    def parallel_twin_primes(limit, num_shards=4, kmax=4):
        manager = multiprocessing.Manager()
        output = manager.Queue()
        shard_bounds = [(i * limit // num_shards, (i + 1) * limit // num_shards) for i in range(num_shards)]
        jobs = [multiprocessing.Process(target=shard_worker, args=(start, end, kmax, output))
                for start, end in shard_bounds]
        for job in jobs: job.start()
        for job in jobs: job.join()
        all_primes = []
        while not output.empty():
            all_primes.extend(output.get())
        return sorted(all_primes)

    # === Module 3: Sophie Germain & Cunningham Chains ===

    def sophie_germain_primes(primes):
        return [(p, 2 * p + 1) for p in primes if isprime(2 * p + 1)]

    def cunningham_chains(primes, max_len=4):
        chains = []
        used = set()
        for p in primes:
            chain = [p]
            while True:
                nxt = 2 * chain[-1] + 1
                if isprime(nxt):
                    chain.append(nxt)
                else:
                    break
            if len(chain) >= max_len:
                chains.append(tuple(chain))
        return chains

    # === Module 4: Entropy Tensor from Glyphs ===

    def normalize(vec, precision=8):
        s = sum(abs(x) for x in vec) or 1.0
        return [round(x / s, precision) for x in vec]

    def harmonic_glyph(prime_pair, H=0.35, t=None):
        p, q = prime_pair
        t = t if t else time.time()
        U = normalize([p, q])
        F = normalize([math.sqrt(p), math.sqrt(q)])
        return {
            "glyph_id": str(uuid.uuid4()),
            "U_k": U,
            "F_Q": F,
            "H": round(H, 8),
            "time_signal": round(math.sin(t) + math.cos(t), 6),
            "coherence_id": hash(tuple(U + F + [H]))
        }

    def entropy_tensor(glyphs):
        from numpy import array, var, mean
        U_matrix = array([g["U_k"] for g in glyphs])
        F_matrix = array([g["F_Q"] for g in glyphs])
        return {
            "U_var": var(U_matrix, axis=0).tolist(),
            "F_var": var(F_matrix, axis=0).tolist(),
            "U_mean": mean(U_matrix, axis=0).tolist(),
            "F_mean": mean(F_matrix, axis=0).tolist()
        }

    # === Module 5: Hardy–Littlewood Integral and Error ===

    def hardy_littlewood_pi2(x):
        integrand = lambda t: 1 / (math.log(t) ** 2) if t > 2 else 0
        C2 = 0.6601618  # Twin prime constant
        integral, _ = quad(integrand, 2, x)
        return int(2 * C2 * integral)

    def relative_error(empirical, x):
        expected = hardy_littlewood_pi2(x)
        return (empirical - expected) / expected

    # === Example Execution ===

    if __name__ == "__main__":
        LIMIT = 10**6
        print("Finding BBP twin primes...")
        twin_primes = parallel_twin_primes(LIMIT, num_shards=4)
        print(f"→ Found {len(twin_primes)} twin primes below {LIMIT}")

        print("Detecting Sophie Germain primes...")
        sophies = sophie_germain_primes([p for p, _ in twin_primes])
        print(f"→ Found {len(sophies)} Sophie Germain primes")

        print("Detecting Cunningham chains...")
        cunningham = cunningham_chains([p for p, _ in twin_primes])
        print(f"→ Found {len(cunningham)} Cunningham chains")

        print("Generating entropy tensor...")
        glyphs = [harmonic_glyph(pair) for pair in twin_primes[:1000]]
        tensor = entropy_tensor(glyphs)
        print("Entropy Tensor Summary:", tensor)

        print("Comparing with Hardy–Littlewood estimate...")
        empirical_count = len(twin_primes)
        err = relative_error(empirical_count, LIMIT)
        print(f"Relative Error at x={LIMIT}: {err:.6%}")


In [ ]:
# SOURCE: MergedNotebook.ipynb#c473
def tokenize(text):
    """
    Extract meaningful tokens from text.
    Preserves technical terms, math notation, Nexus symbols.
    """
    tokens = []
    text_lower = text.lower()
    
    # Greek letters
    greek = re.findall(r'[αβγδεζηθικλμνξοπρστυφχψω]+', text_lower)
    tokens.extend(greek)
    
    # Mathematical symbols as concepts
    if 'π' in text or 'pi' in text_lower:
        tokens.append('pi')
    if 'φ' in text or 'phi' in text_lower:
        tokens.append('phi')
    if 'ψ' in text or 'psi' in text_lower:
        tokens.append('psi')
    if '∆' in text or 'delta' in text_lower:
        tokens.append('delta')
    if 'σ' in text or 'sigma' in text_lower:
        tokens.append('sigma')
    if '→' in text:
        tokens.append('arrow_transform')
    if '⊕' in text:
        tokens.append('xor_symbol')
    if '↻' in text:
        tokens.append('recursive_symbol')
    
    # snake_case terms (preserve compound meaning)
    snake_terms = re.findall(r'[a-z]+(?:_[a-z]+)+', text_lower)
    tokens.extend(snake_terms)
    
    # CamelCase terms
    camel = re.findall(r'[A-Z][a-z]+(?:[A-Z][a-z]+)+', text)
    tokens.extend([c.lower() for c in camel])
    
    # Regular words (3+ chars)
    words = re.findall(r'\b[a-z]{3,}\b', text_lower)
    tokens.extend(words)
    
    # Significant numbers
    numbers = re.findall(r'\b(?:256|512|1024|2048|4096|64|128|32|16|8)\b', text)
    tokens.extend([f'num_{n}' for n in numbers])
    
    return tokens

# Test it
test_text = "The SHA-256 hash function uses recursive_fold operations with π alignment"
print("Test tokens:", tokenize(test_text))

In [ ]:
# SOURCE: MergedNotebook.ipynb#c494
# sha256_harmonic_probe.py
# Single-block SHA-256 round tracer with Mark1/Samson-aligned metrics.
# Python 3.9+

from typing import List, Tuple

# --- Bit ops (32-bit wrap) ---
MASK32 = 0xFFFFFFFF
def rotr(x, n): return ((x >> n) | ((x & MASK32) << (32 - n))) & MASK32
def shr(x, n):  return (x >> n) & MASK32

def Σ0(x): return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)
def Σ1(x): return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)
def σ0(x): return rotr(x, 7) ^ rotr(x, 18) ^ shr(x, 3)
def σ1(x): return rotr(x,17) ^ rotr(x, 19) ^ shr(x,10)

def Ch(x,y,z):  return (x & y) ^ ((~x) & z)
def Maj(x,y,z): return (x & y) ^ (x & z) ^ (y & z)

# SHA-256 constants (FIPS 180-4)
K = [
  0x428a2f98,0x71374491,0xb5c0fbcf,0xe9b5dba5,0x3956c25b,0x59f111f1,0x923f82a4,0xab1c5ed5,
  0xd807aa98,0x12835b01,0x243185be,0x550c7dc3,0x72be5d74,0x80deb1fe,0x9bdc06a7,0xc19bf174,
  0xe49b69c1,0xefbe4786,0x0fc19dc6,0x240ca1cc,0x2de92c6f,0x4a7484aa,0x5cb0a9dc,0x76f988da,
  0x983e5152,0xa831c66d,0xb00327c8,0xbf597fc7,0xc6e00bf3,0xd5a79147,0x06ca6351,0x14292967,
  0x27b70a85,0x2e1b2138,0x4d2c6dfc,0x53380d13,0x650a7354,0x766a0abb,0x81c2c92e,0x92722c85,
  0xa2bfe8a1,0xa81a664b,0xc24b8b70,0xc76c51a3,0xd192e819,0xd6990624,0xf40e3585,0x106aa070,
  0x19a4c116,0x1e376c08,0x2748774c,0x34b0bcb5,0x391c0cb3,0x4ed8aa4a,0x5b9cca4f,0x682e6ff3,
  0x748f82ee,0x78a5636f,0x84c87814,0x8cc70208,0x90befffa,0xa4506ceb,0xbef9a3f7,0xc67178f2
]

H0 = [  # initial hash values
  0x6a09e667,0xbb67ae85,0x3c6ef372,0xa54ff53a,
  0x510e527f,0x9b05688c,0x1f83d9ab,0x5be0cd19
]

def pad_single_block(msg_bytes: bytes) -> bytes:
    """Pad for a single 512-bit block (works if len(msg)<56)."""
    L = len(msg_bytes)
    assert L < 56, "This probe handles single-block messages (<56 bytes)."
    ml_bits = L * 8
    x = msg_bytes + b'\x80'
    x += b'\x00' * (56 - len(x))
    x += ml_bits.to_bytes(8, 'big')
    return x

def words32(block: bytes) -> List[int]:
    return [int.from_bytes(block[i:i+4], 'big') for i in range(0, 64, 4)]

def schedule(W0_15: List[int]) -> List[int]:
    W = W0_15[:] + [0]*48
    for t in range(16, 64):
        W[t] = (σ1(W[t-2]) + W[t-7] + σ0(W[t-15]) + W[t-16]) & MASK32
    return W

def hamming256(state_a: Tuple[int,...], state_b: Tuple[int,...]) -> int:
    flips = 0
    for x,y in zip(state_a, state_b):
        flips += ((x ^ y) & MASK32).bit_count()
    return flips

def sha256_field_probe(msg: bytes):
    block = pad_single_block(msg)
    W = schedule(words32(block))

    a,b,c,d,e,f,g,h = H0
    states = []
    # pseudo-round 0 state (pre-round) for baseline flip calc
    states.append((a,b,c,d,e,f,g,h))

    for t in range(64):
        T1 = (h + Σ1(e) + Ch(e,f,g) + K[t] + W[t]) & MASK32
        T2 = (Σ0(a) + Maj(a,b,c)) & MASK32
        h = g
        g = f
        f = e
        e = (d + T1) & MASK32
        d = c
        c = b
        b = a
        a = (T1 + T2) & MASK32
        states.append((a,b,c,d,e,f,g,h))

    # digest (after adding to H0)
    A,B,C,D,E,F,G,H_ = states[-1]
    A = (A + H0[0]) & MASK32; B = (B + H0[1]) & MASK32
    C = (C + H0[2]) & MASK32; D = (D + H0[3]) & MASK32
    E = (E + H0[4]) & MASK32; F = (F + H0[5]) & MASK32
    G = (G + H0[6]) & MASK32; Hh= (H_+ H0[7]) & MASK32
    digest = ''.join(f'{w:08x}' for w in [A,B,C,D,E,F,G,Hh])

    # per-round bit-flip “energy”
    E_round = []
    for r in range(1, len(states)):
        E_round.append(hamming256(states[r], states[r-1]))

    # harmony H: fraction of rounds that reduce flips vs previous
    stabilizing = sum(1 for i in range(1, len(E_round)) if E_round[i] < E_round[i-1])
    H_mark1 = stabilizing / (len(E_round)-1)

    # Samson summaries: level and derivative
    # S0(r)=E_round[r]; D(r)=E_round[r]-E_round[r-1]
    deriv = [E_round[i]-E_round[i-1] for i in range(1, len(E_round))]
    S_summary = {
        "E_mean": sum(E_round)/len(E_round),
        "E_min": min(E_round),
        "E_max": max(E_round),
        "dE_mean": sum(deriv)/len(deriv),
        "dE_neg_fraction": sum(1 for x in deriv if x < 0) / len(deriv)
    }

    return {
        "digest": digest,
        "H_mark1": H_mark1,                # target band ~0.35
        "stabilizing_steps": stabilizing,
        "total_transitions": len(E_round)-1,
        "samson": S_summary,
        # Optionally expose E_round or states for deeper analysis
        # "E_round": E_round,
        # "states": states,
    }

if __name__ == "__main__":
    for s in [b"Hello", b"hello", b"Hello."]:
        out = sha256_field_probe(s)
        print(f"input={s!r}")
        print("digest =", out["digest"])
        print("H_mark1 =", round(out["H_mark1"], 3),
              "(stabilizing/steps =", f"{out['stabilizing_steps']}/{out['total_transitions']})")
        print("Samson:", out["samson"])
        print("-"*60)


In [ ]:
# SOURCE: MergedNotebook.ipynb#c495
# --- Add to sha256_harmonic_probe.py (below existing code) ---

def run_punctuation_sweep(base: bytes):
    tests = {
        "pause":   [b".", b",", b":"],
        "jump":    [b";", b"!", b"?"],
        "group":   [b"()", b"[]", b"{}"],
        "boundary":[b" ", b"\n", b"\r\n"],
    }

    def probe(lbl, s: bytes):
        out = sha256_field_probe(s)
        return (lbl, s, out["H_mark1"], 
                out["samson"]["E_mean"], 
                out["samson"]["dE_mean"],
                out["samson"]["dE_neg_fraction"])

    rows = []
    # appends
    for cls, plist in tests.items():
        for p in plist:
            rows.append(probe(f"{cls}:append:{p!r}", base + p))

    # balanced placement at 4-byte boundary (if length allows)
    idx4 = ((len(base) + 3) // 4) * 4  # next 4B boundary (>= len(base))
    for p in [b"()", b"[]", b"{}"]:
        s = base + b"X" * (idx4 - len(base)) + p  # pad with benign bytes to hit boundary
        rows.append(probe(f"group:4B_boundary:{p!r}", s))

    # print results
    print("label, input, H, E_mean, dE_mean, dE_neg_fraction")
    for r in rows:
        label, s, H, Em, dEm, fracNeg = r
        print(f"{label}, {s!r}, {H:.3f}, {Em:.3f}, {dEm:.3f}, {fracNeg:.3f}")

if __name__ == "__main__":
    # existing demo
    for s in [b"Hello", b"hello", b"Hello."]:
        out = sha256_field_probe(s)
        print(f"input={s!r}")
        print("digest =", out["digest"])
        print("H_mark1 =", round(out["H_mark1"], 3),
              "(stabilizing/steps =", f"{out['stabilizing_steps']}/{out['total_transitions']})")
        print("Samson:", out["samson"])
        print("-"*60)

    # NEW: punctuation sweep off the neutral seed
    run_punctuation_sweep(b"Hello")


In [ ]:
# SOURCE: MergedNotebook.ipynb#c582
import numpy as np
from math import ceil, log2

def zero_pad_to_block(seed, block_size):
    """
    Pad the binary representation of the seed with zeros to fit the block size.
    """
    seed_binary = "".join(f"{ord(c):08b}" for c in seed)

    # Pad with zeros to fit the block size
    padded_binary = seed_binary
    while len(padded_binary) % block_size != 0:
        padded_binary += "0"

    return [int(bit) for bit in padded_binary]  # Return as a list of bits

def determine_container_size(data_length):
    """
    Determine the smallest base and container size that fits the data length.
    """
    base = int(ceil(log2(data_length)))
    container_size = 2 ** base
    return base, container_size

def binary_to_base(data, base):
    """
    Convert binary data to a specified base.
    """
    value = int("".join(str(bit) for bit in data), 2)  # Convert binary to integer
    result = []
    while value > 0:
        result.append(value % base)
        value //= base
    return result[::-1]  # Return reversed for proper order

def base_to_binary(data, base):
    """
    Convert data in a specified base back to binary.
    """
    value = sum(d * (base ** i) for i, d in enumerate(reversed(data)))  # Convert base to integer
    binary = bin(value)[2:]  # Convert integer to binary string
    return [int(bit) for bit in binary.zfill(len(data) * int(log2(base)))]  # Ensure proper bit length

def binary_to_hex(binary_data):
    """
    Convert binary data to hexadecimal representation.
    """
    binary_string = "".join(map(str, binary_data))
    hex_string = hex(int(binary_string, 2))[2:].upper()  # Convert to hex and remove "0x" prefix
    return hex_string

def binary_to_text(binary_data):
    """
    Convert binary data back to text.
    """
    # Strip leading zeros
    binary_string = "".join(map(str, binary_data)).lstrip("0")

    # Convert binary string to text
    chars = [chr(int(binary_string[i:i+8], 2)) for i in range(0, len(binary_string), 8)]
    return "".join(chars).rstrip("\x00")  # Strip null characters added during padding

def compress_large_input(seed, initial_base=16, final_base=18):
    """
    Compress large input dynamically by adjusting container size and base.
    """
    # Determine container size dynamically
    padded_data = zero_pad_to_block(seed, block_size=8192)
    data_length = len(padded_data)
    base, container_size = determine_container_size(data_length)

    print(f"Input Length: {data_length}, Determined Base: {base}, Container Size: {container_size}")

    # Initialize the fixed container
    container = [0] * container_size
    container[:len(padded_data)] = padded_data[:container_size]  # Fill initial container

    print("\nInitial Container (Binary):", container)

    # Iteratively perform base conversions
    current_data = container
    for b in range(initial_base, final_base + 1):
        # Step 1: Convert to next base
        converted_base = binary_to_base(current_data, b)
        print(f"\nBase-{b} Representation:", converted_base)

        # Step 2: Convert back to binary
        current_data = base_to_binary(converted_base, b)
        print(f"Binary Representation after Base-{b} Conversion:", current_data)

        # Step 3: Store the converted data in the container
        container = [0] * container_size  # Reset container
        container[:len(current_data)] = current_data[:container_size]  # Ensure it fits
        print(f"Container after Base-{b} Conversion:", container)

    # Convert final binary container to hexadecimal
    final_hex = binary_to_hex(container)
    print("\nFinal Compressed Data in Hexadecimal:", final_hex)

    return container, final_hex

def decompress_large_input(compressed_data, initial_base=18, final_base=16):
    """
    Decompress large input using dynamically adjusted container size and base.
    """
    current_data = compressed_data

    for b in range(initial_base, final_base - 1, -1):
        # Step 1: Convert binary to base
        converted_base = binary_to_base(current_data, b)
        print(f"\nBase-{b} Representation during Decompression:", converted_base)

        # Step 2: Convert base back to binary
        current_data = base_to_binary(converted_base, b)
        print(f"Binary Representation after Base-{b} Conversion:", current_data)

    # Strip trailing zeros to get the original data
    decompressed_data = current_data[:len(current_data) - current_data[::-1].index(1)]
    print("\nDecompressed Data (Binary):", decompressed_data)

    # Convert binary to text
    original_text = binary_to_text(decompressed_data)
    print("\nDecompressed Text:", original_text)

    return original_text

# Example usage
seed = "Watson, I need you"
compressed_data, compressed_hex = compress_large_input(seed)

print("\nFinal Hash in Hexadecimal:", compressed_hex)

# Decompress the data
decompressed_text = decompress_large_input(compressed_data)



In [ ]:
# SOURCE: MergedNotebook.ipynb#c583
import numpy as np
from math import ceil, log2

def zero_pad_to_block(seed, block_size):
    """
    Pad the binary representation of the seed with zeros to fit the block size.
    """
    seed_binary = "".join(f"{ord(c):08b}" for c in seed)

    # Pad with zeros to fit the block size
    padded_binary = seed_binary
    while len(padded_binary) % block_size != 0:
        padded_binary += "0"

    return [int(bit) for bit in padded_binary]  # Return as a list of bits

def determine_container_size(data_length):
    """
    Determine the smallest base and container size that fits the data length.
    """
    base = int(ceil(log2(data_length)))
    container_size = 2 ** base
    return base, container_size

def binary_to_base(data, base):
    """
    Convert binary data to a specified base.
    """
    value = int("".join(str(bit) for bit in data), 2)  # Convert binary to integer
    result = []
    while value > 0:
        result.append(value % base)
        value //= base
    return result[::-1]  # Return reversed for proper order

def base_to_binary(data, base):
    """
    Convert data in a specified base back to binary.
    """
    value = sum(d * (base ** i) for i, d in enumerate(reversed(data)))  # Convert base to integer
    binary = bin(value)[2:]  # Convert integer to binary string
    return [int(bit) for bit in binary.zfill(len(data) * int(log2(base)))]  # Ensure proper bit length

def binary_to_hex(binary_data):
    """
    Convert binary data to hexadecimal representation.
    """
    binary_string = "".join(map(str, binary_data))
    hex_string = hex(int(binary_string, 2))[2:].upper()  # Convert to hex and remove "0x" prefix
    return hex_string

def binary_to_text(binary_data):
    """
    Convert binary data back to text.
    """
    # Strip leading zeros
    binary_string = "".join(map(str, binary_data)).lstrip("0")

    # Convert binary string to text
    chars = [chr(int(binary_string[i:i+8], 2)) for i in range(0, len(binary_string), 8)]
    return "".join(chars).rstrip("\x00")  # Strip null characters added during padding

def compress_large_input(seed, initial_base=16, final_base=18):
    """
    Compress large input dynamically by adjusting container size and base.
    """
    # Determine container size dynamically
    padded_data = zero_pad_to_block(seed, block_size=512)
    data_length = len(padded_data)
    base, container_size = determine_container_size(data_length)

    print(f"Input Length: {data_length}, Determined Base: {base}, Container Size: {container_size}")

    # Initialize the fixed container
    container = [0] * container_size
    container[:len(padded_data)] = padded_data[:container_size]  # Fill initial container

    print("\nInitial Container (Binary):", container)

    # Iteratively perform base conversions
    current_data = container
    for b in range(initial_base, final_base + 1):
        # Step 1: Convert to next base
        converted_base = binary_to_base(current_data, b)
        print(f"\nBase-{b} Representation:", converted_base)

        # Step 2: Convert back to binary
        current_data = base_to_binary(converted_base, b)
        print(f"Binary Representation after Base-{b} Conversion:", current_data)

        # Step 3: Store the converted data in the container
        container = [0] * container_size  # Reset container
        container[:len(current_data)] = current_data[:container_size]  # Ensure it fits
        print(f"Container after Base-{b} Conversion:", container)

    # Convert final binary container to hexadecimal
    final_hex = binary_to_hex(container)
    print("\nFinal Compressed Data in Hexadecimal:", final_hex)

    return container, final_hex

def decompress_large_input(compressed_data, initial_base=18, final_base=16):
    """
    Decompress large input using dynamically adjusted container size and base.
    """
    current_data = compressed_data

    for b in range(initial_base, final_base - 1, -1):
        # Step 1: Convert binary to base
        converted_base = binary_to_base(current_data, b)
        print(f"\nBase-{b} Representation during Decompression:", converted_base)

        # Step 2: Convert base back to binary
        current_data = base_to_binary(converted_base, b)
        print(f"Binary Representation after Base-{b} Conversion:", current_data)

    # Strip trailing zeros to get the original data
    decompressed_data = current_data[:len(current_data) - current_data[::-1].index(1)]
    print("\nDecompressed Data (Binary):", decompressed_data)

    # Convert binary to text
    original_text = binary_to_text(decompressed_data)
    print("\nDecompressed Text:", original_text)

    return original_text

# Example usage
seed = "test"
compressed_data, compressed_hex = compress_large_input(seed)

print("\nFinal Hash in Hexadecimal:", compressed_hex)

# Decompress the data
decompressed_text = decompress_large_input(compressed_data)



In [ ]:
# SOURCE: MergedNotebook.ipynb#c764
# Re-import necessary modules after code execution state reset
import plotly.graph_objects as go
import numpy as np
import plotly.io as pio
import os

# Set the output path on the D: drive
output_path = "D:\\recursive_pi_triangle.html"

# Triangle vertices
triangle = np.array([
    [0, 0],
    [1, 0],
    [0.5, np.sqrt(3)/2],
    [0, 0]  # closing the triangle
])

# Bouncing inside the triangle
def reflect(p, v, a, b):
    """Reflect vector v at point p off the edge from a to b."""
    edge = b - a
    edge_norm = np.array([-edge[1], edge[0]])
    edge_norm = edge_norm / np.linalg.norm(edge_norm)
    v_reflected = v - 2 * np.dot(v, edge_norm) * edge_norm
    return v_reflected

# Initial point and velocity
point = np.array([0.1, 0.1])
velocity = np.array([0.03, 0.05])
points = [point.copy()]
num_bounces = 500

for _ in range(num_bounces):
    next_point = point + velocity

    for i in range(3):
        a = triangle[i]
        b = triangle[i+1]
        edge = b - a
        edge_norm = np.array([-edge[1], edge[0]])
        edge_norm = edge_norm / np.linalg.norm(edge_norm)

        to_line = np.dot((next_point - a), edge_norm)
        from_line = np.dot((point - a), edge_norm)

        if np.sign(to_line) != np.sign(from_line):
            velocity = reflect(point, velocity, a, b)
            next_point = point + velocity
            break

    point = next_point
    points.append(point.copy())

points = np.array(points)

# Create animation with plotly
fig = go.Figure()

fig.add_trace(go.Scatter(x=triangle[:, 0], y=triangle[:, 1],
                         mode="lines", name="Triangle"))

fig.add_trace(go.Scatter(x=points[:, 0], y=points[:, 1],
                         mode="lines", name="Path",
                         line=dict(color="cyan")))

fig.update_layout(
    title="Recursive π Triangle Reflection",
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    showlegend=False,
    plot_bgcolor='black',
    paper_bgcolor='black',
    width=600,
    height=600
)

# Save as HTML
pio.write_html(fig, file=output_path, auto_open=False)
output_path


In [ ]:
# SOURCE: MergedNotebook.ipynb#c828
import matplotlib.artist  # Nexus patch: ensure matplotlib.artist exists
import numpy as np
import hashlib
import matplotlib.pyplot as plt

def krrb(R0, H, F, B_func, steps=1000, dt=0.5):
    R = np.zeros(steps + 1, dtype=np.complex128)
    R[0] = R0
    for k in range(steps):
        Bk = B_func(k)
        R[k + 1] = R[k] * np.exp(H * F * dt) * np.prod(Bk)
    return R

def sha_branch_factors(data, n_branches=4):
    h = hashlib.sha256(data.encode("utf8")).digest()
    vals = np.frombuffer(h, dtype=np.uint8).astype(np.float64)
    vals = vals[:n_branches] / 255.0
    B = 0.9 + 0.2 * vals
    return B

# Byte1 as string
byte1_str = "314159265"

# Compute fixed branch factors from Byte1
fixed_B = sha_branch_factors(byte1_str, n_branches=4)

def constant_B_func(k):
    return fixed_B

# Parameters
R0 = 1.0 + 0j
H = np.pi / 9
F = 1.0
steps = 1000
dt = 0.5

# Run simulation
R_byte1 = krrb(R0, H, F, constant_B_func, steps=steps, dt=dt)

# Plot magnitude and phase
t = np.arange(steps + 1) * dt

plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(t, np.abs(R_byte1))
plt.title('Magnitude |R(t)| - Byte1 Input')
plt.ylabel('|R|')
plt.grid(True)

plt.subplot(2, 1, 2)
plt.plot(t, np.angle(R_byte1))
plt.title('Phase arg(R(t)) - Byte1 Input')
plt.xlabel('Time t')
plt.ylabel('Phase (rad)')
plt.grid(True)

plt.tight_layout()
plt.show()

# Print key values
print("Branch factors from Byte1:", fixed_B)
print("Effective growth rate per step:", np.exp(H * F * dt) * np.prod(fixed_B))
print("Final |R|:", np.abs(R_byte1[-1]))
print("Final phase:", np.angle(R_byte1[-1]))

## Quarantined cells (did not run)
| cell | error | note |
|---:|---|---|
| 1 | NameError: name '__file__' is not defined | missing dependency / variable not defined in this subset |
| 5 | ModuleNotFoundError: No module named 'transformers' | optional dependency not installed |
| 8 | NameError: name 'TrainerCallback' is not defined | missing dependency / variable not defined in this subset |
| 9 | NameError: name 'dataset' is not defined | missing dependency / variable not defined in this subset |
| 10 | LookupError: 
*********************************************************** | needs refactor |
| 11 | FileNotFoundError: [Errno 2] No such file or directory: 'd://nexus_helix.pdb' | requires local file path |
| 12 | ModuleNotFoundError: No module named 'transformers' | optional dependency not installed |
| 13 | NameError: name 'TrainerCallback' is not defined | missing dependency / variable not defined in this subset |
| 14 | NameError: name 'dataset' is not defined | missing dependency / variable not defined in this subset |
| 16 | TypeError: unsupported operand type(s) for -: 'list' and 'list' | needs refactor |
| 17 | TIMEOUT: timeout | likely heavy loop; needs guarding/refactor |
| 18 | NameError: name 'mean_ratio' is not defined | missing dependency / variable not defined in this subset |
| 20 | TIMEOUT: timeout | likely heavy loop; needs guarding/refactor |
| 21 | SyntaxError: invalid syntax (<cell>, line 2) | needs refactor |
| 29 | SyntaxError: bytes can only contain ASCII literal characters (<cell>, lin | needs refactor |
| 30 | SyntaxError: invalid character '’' (U+2019) (<cell>, line 3) | needs refactor |
| 32 | IndexError: index 4 is out of bounds for axis 0 with size 4 | needs refactor |
| 34 | TIMEOUT: timeout | likely heavy loop; needs guarding/refactor |
